# Federated Analytics

This notebook serves as a base example to how to explore on datasets by calculating some different analytics functions, without directly initializing a model and a training. We will see how the mean is calculated on nodes which contain the ADNI Dataset.

### Setting the node up
Before running this notebook we configure 2 nodes: <br/>
* **Node 1 :**
  * Add a dataset `fedbiomed node -p my-first-node dataset add`
  * Select option, choose the name, tags and description of the dataset
  * Check that your data has been added by executing `fedbiomed node -p my-first-node list`
  * Run the node using `fedbiomed node -p my-first-node start`. <br/>

* **Node 2 :**
  * Repeat the same steps for `my-second-node`

Wait until you get `Starting task manager`, it means node is online.

### Setting up the experiment

We set up an experiment, in order to select the nodes which contain the datasets and perform federated analytics on them.

## Tabular Dataset

For a tabular dataset you can input which columns you want to compute the analytics on. The columns are given as a list, which can be a list of column names or column indexes.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment

# The `Experiment` scans all connected nodes for datasets matching the `'tabular'` tag. 
# Two nodes are discovered and registered for the experiment.
tags = ['tabular']
exp = Experiment(tags=tags)

In [ ]:
# Both nodes host CSV datasets with the same 7 columns and compatible types.
# Node 1 has 10 668 rows, Node 2 has 17 965 rows. 
# This metadata is returned by nodes without exposing any raw data.
exp.training_data().data()

In [ ]:
# `fetch_stats('mean')` sends an analytics request to both nodes. 
# Each node computes the mean and count for every numeric column locally and returns them.
# Results are stored in an `FAResult` object and cached.
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids

In [ ]:
result.schema

In [ ]:
result.node_stats()

In [ ]:
# `available_stats()` lists statistics present in the node replies (`count`, `mean`).
# `computable_stats()` additionally includes `sum`, which can be derived locally from `mean × count` without re-querying nodes.
print(result.available_stats())
print(result.computable_stats())

In [ ]:
# `global_stats('mean')` combines per-node means into a single count-weighted average across all nodes.
result.global_stats("mean")

In [ ]:
# `exp.analytics.mean()` is a shorthand that calls `fetch_stats('mean')` then `global_stats('mean')`. 
# The log confirms the result is served from cache — nodes are not contacted again.
exp.analytics.mean()

In [ ]:
# `min` and `max` are not recognized statistics. 
# `fetch_stats` raises a `FedbiomedError` immediately, before contacting any node. 
# The previously fetched `result` object is unaffected and still holds the mean data.
result = exp.analytics.fetch_stats(["min", "max"])

In [ ]:
result.node_stats()

In [ ]:
# `'histogram'` passes client-side validation but is rejected by both nodes, which do not yet implement it.
# The error is fatal — no partial results are saved and the request does not overwrite the cached result.
result = exp.analytics.fetch_stats("histogram")

In [ ]:
result = exp.analytics.fetch_stats_with_args(stats_args={
    "price": {
        "histogram": {
            "bin_edges": [100,125,150,175,200,250]
        }
    }
})

In [ ]:
# After two failed histogram requests, `result` still holds the mean data from the original successful call.
# Failed requests never overwrite the cached result.
result.node_stats()

## Mnist Dataset

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment

tags = ['#MNIST', '#dataset']
exp = Experiment(tags=tags)
exp.training_data().data()

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids

In [ ]:
# The schema is empty because MNIST has no named tabular columns. 
# With no columns to operate on, `available_stats`, `computable_stats`, and `node_stats` are all empty. 
# No error is raised — the request simply returns nothing.
print(result.schema)
print(result.available_stats())
print(result.computable_stats())

In [ ]:
result.node_stats()

## Medical Folder Dataset

The IXI dataset combines NIfTI image modalities (`T1`, `T2`, `label`) with a demographics CSV. Analytics can only be computed on named tabular columns; image modalities appear in the schema but produce no statistics.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment

tags = ['ixi']
exp = Experiment(tags=tags)
exp.training_data().data()

In [ ]:
result = exp.analytics.fetch_stats("mean")

In [ ]:
result.node_ids

In [ ]:
# The schema reflects the nested structure: `demographics` has named sub-columns (e.g. `AGE`, `HEIGHT`),
# while `T1`, `T2`, and `label` are image modalities with no sub-columns.
result.schema

In [ ]:
result.available_stats()

In [ ]:
result.node_stats()

Only numeric demographics columns have valid stats. `DOB`, `STUDY_DATE`, and `SITE_NAME` return `nan` with `count: 0` because they are non-numeric (dates or text). Image modality keys (`T1`, `T2`, `label`) are empty — per-voxel statistics are not computed.

## Secure Aggregation with Federated Analytics

Secure Aggregation (SecAgg) prevents the researcher from seeing individual node statistics. Each node encrypts its output before sending; the researcher decrypts only the cross-node aggregate.

Two schemes are supported:

| Scheme | Requirement | Minimum rounds |
|--------|-------------|---------------|
| **LOM** (default) | ≥ 1 node, pairwise ChaCha20 PRF masks | 1 |
| **JLS** (Joye-Libert) | ≥ 2 nodes, homomorphic encryption | 3 |

When SecAgg is active, `FAResult.node_ids` returns `["__secagg__"]` — the researcher sees only the aggregate, never per-node values.

**Node prerequisites**: each node must have `secure_aggregation = True` in its `[security]` config section. Nodes with `force_secure_aggregation = True` will reject plaintext FA requests.

In [ ]:
from fedbiomed.researcher.federated_workflows import Experiment, FederatedAnalytics
from fedbiomed.researcher.secagg import SecureAggregation, SecureAggregationSchemes

# Discover nodes and initialise shared transport infrastructure used by all scenarios below.
tags = ['tabular']
exp = Experiment(tags=tags)
exp.training_data().data()

### Scenario 1 — LOM scheme (default)

LOM is the default and lightest scheme — it uses pairwise ChaCha20 PRF masks and works with a single node.
`clipping_range` controls the maximum absolute stat value that survives encryption without overflow; the default `1000.0` covers typical counts, means, and variances.

In [ ]:
secagg_lom = SecureAggregation(active=True)  # LOM is the default scheme

fa_lom = FederatedAnalytics(
    fds=exp.training_data(),
    experiment_id=exp.id,
    researcher_id=exp.researcher_id,
    reqs=exp.requests,
    experimentation_folder=exp.experimentation_folder(),
    secagg=secagg_lom,
    clipping_range=1000,  # raise this if columns can exceed ±1000
)

In [ ]:
# SecAgg key exchange (setup) happens automatically on the first request.
# Each node encrypts its stats vector before sending; the researcher decrypts only the sum.
result_lom = fa_lom.fetch_stats("mean")

In [ ]:
# With SecAgg active, individual node identities are hidden.
# Only the aggregate virtual node '__secagg__' is visible.
result_lom.node_ids

In [ ]:
print("Schema:      ", result_lom.schema)

# Per-node raw values are not accessible; only the decrypted sum is stored.
print("Aggregate:   ", result_lom.node_stats("__secagg__"))

# global_stats() API is unchanged — it aggregates across all entries in the result.
print("Global mean: ", result_lom.global_stats("mean"))

### Scenario 2 — JLS scheme

JLS (Joye-Libert) uses additively homomorphic encryption and requires **≥ 2 nodes**.
It enforces a **minimum of 3 rounds** — the first two `fetch_stats` calls with a fresh JLS context complete the protocol bootstrap; only the third call (and beyond) produces a correctly decryptable result.

In [ ]:
secagg_jls = SecureAggregation(
    active=True,
    scheme=SecureAggregationSchemes.JOYE_LIBERT,
)

fa_jls = FederatedAnalytics(
    fds=exp.training_data(),
    experiment_id=exp.id,
    researcher_id=exp.researcher_id,
    reqs=exp.requests,
    experimentation_folder=exp.experimentation_folder(),
    secagg=secagg_jls,
)

# JLS enforces a minimum of 3 rounds before the first correct decryption.
# The first two calls advance the round counter through the protocol bootstrap.
fa_jls.fetch_stats("count")     # round 1 — bootstrap
fa_jls.fetch_stats("variance")  # round 2 — bootstrap
result_jls = fa_jls.fetch_stats("mean")  # round 3 — first valid result
result_jls.global_stats("mean")

### Scenario 3 — Custom clipping range

`clipping_range` sets the maximum absolute stat value that can be represented after encryption.
Values outside `[-clipping_range, +clipping_range]` overflow silently and produce incorrect aggregates.
Set it to the expected upper bound for the columns in your dataset — the default `1000.0` covers typical counts and means, but income, pixel intensity, or other large-magnitude columns need a higher value.

In [ ]:
fa_large_range = FederatedAnalytics(
    fds=exp.training_data(),
    experiment_id=exp.id,
    researcher_id=exp.researcher_id,
    reqs=exp.requests,
    experimentation_folder=exp.experimentation_folder(),
    secagg=SecureAggregation(active=True),
    clipping_range=200_000,
)

result_large = fa_large_range.fetch_stats("mean")
result_large.global_stats("mean")

### Scenario 4 — Round counter and result caching

Each `fetch_stats` call that contacts nodes increments `_fa_round_counter`.
The counter is embedded in `secagg_arguments` so nodes can reject replayed or out-of-order requests.
The result cache works identically to plaintext mode: a second call with the same stats skips the network round and does not increment the counter.

In [ ]:
fa_rounds = FederatedAnalytics(
    fds=exp.training_data(),
    experiment_id=exp.id,
    researcher_id=exp.researcher_id,
    reqs=exp.requests,
    experimentation_folder=exp.experimentation_folder(),
    secagg=SecureAggregation(active=True),
)

fa_rounds.fetch_stats("count")      # round 1
print("After count       — fa_round_counter:", fa_rounds._fa_round_counter)

fa_rounds.fetch_stats("mean")       # round 2: mean primitives not yet cached
print("After mean        — fa_round_counter:", fa_rounds._fa_round_counter)

fa_rounds.fetch_stats("mean")       # cache hit — no new round
print("After mean (cache) — fa_round_counter:", fa_rounds._fa_round_counter)

# Both count and mean primitives are now merged under '__secagg__'.
fa_rounds.fetch_stats("mean").global_stats("mean")

### Scenario 5 — Inactive SecAgg (plaintext path)

With `active=False`, FA behaves exactly as before: nodes send plaintext stats, results are keyed by real node IDs, and per-node values are accessible. Use this to verify that switching SecAgg off doesn't regress existing analytics.

In [ ]:
fa_plain = FederatedAnalytics(
    fds=exp.training_data(),
    experiment_id=exp.id,
    researcher_id=exp.researcher_id,
    reqs=exp.requests,
    experimentation_folder=exp.experimentation_folder(),
    secagg=SecureAggregation(active=False),
)

result_plain = fa_plain.fetch_stats("mean")

# Real node IDs are visible, not '__secagg__'
print("Node IDs:", result_plain.node_ids)
print("Per-node stats:", result_plain.node_stats())
result_plain.global_stats("mean")